<a href="https://colab.research.google.com/github/himanshubhimte69/Parameter_Efficient_Deep_Learning-Class_Selective_Knowledge_Distillation_for_Precision_Agriculture/blob/main/grad_diagram.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# 1. IMPORTS
# ============================================================

import os
import cv2
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from tensorflow.keras.applications.efficientnet import (
    EfficientNetB4,
    preprocess_input
)

from tensorflow.keras.layers import (
    GlobalAveragePooling2D,
    Dense
)

from tensorflow.keras.models import Model

from tensorflow.keras.preprocessing import image

print("TensorFlow:", tf.__version__)


TensorFlow: 2.20.0


In [ ]:
!unzip gradcam_images.zip

Archive:  gradcam_images.zip
   creating: gradcam_images/
  inflating: gradcam_images/gradcam_Train_26.jpg  
  inflating: gradcam_images/gradcam_Train_29.jpg  
  inflating: gradcam_images/gradcam_Train_83.jpg  


In [ ]:
# ============================================================
# 2. SETTINGS
# ============================================================

IMG_SIZE = 380

# ------------------------------------------------------------
# FOLDER CONTAINING TEST IMAGES
# ------------------------------------------------------------

test_folder = "/content/gradcam_images"

# ------------------------------------------------------------
# SAVE DIRECTORY
# ------------------------------------------------------------

save_dir = "/content/gradcam_images"

os.makedirs(
    save_dir,
    exist_ok=True
)


In [ ]:
# ============================================================
# 3. CLASS NAMES
# ============================================================

class_names = [
    "healthy",
    "rust",
    "scab"
]

In [ ]:
# ============================================================
# 4. BUILD BASELINE MODEL
# ============================================================

baseline_base = EfficientNetB4(
    include_top=False,
    weights=None,
    input_shape=(380,380,3)
)

baseline_model = tf.keras.Sequential([
    baseline_base,
    GlobalAveragePooling2D(),
    Dense(
        3,
        activation='softmax'
    )
])

baseline_model.build(
    (None,380,380,3)
)

# ------------------------------------------------------------
# LOAD WEIGHTS
# ------------------------------------------------------------

baseline_model.load_weights(
    "/content/basemodel_model_pp2020.h5",
    by_name=True,
    skip_mismatch=True
)

print("Baseline model loaded")


Baseline model loaded


In [ ]:
# ============================================================
# 5. BUILD KD MODEL
# ============================================================

kd_base = EfficientNetB4(
    include_top=False,
    weights=None,
    input_shape=(380,380,3)
)

student_model_pp2020 = tf.keras.Sequential([
    kd_base,
    GlobalAveragePooling2D(),
    Dense(
        3,
        activation='softmax'
    )
])

student_model_pp2020.build(
    (None,380,380,3)
)

# ------------------------------------------------------------
# LOAD WEIGHTS
# ------------------------------------------------------------

student_model_pp2020.load_weights(
    "/content/student_model_pp2020.h5",
    by_name=True,
    skip_mismatch=True
)

print("KD model loaded")

KD model loaded


In [ ]:
# ============================================================
# INITIALIZE MODELS
# ============================================================

dummy_input = tf.random.normal(
    (1,380,380,3)
)

_ = baseline_model(dummy_input)

_ = student_model_pp2020(dummy_input)

print("Models initialized successfully")

Models initialized successfully


In [ ]:
# ============================================================
# 6. LAST CONVOLUTIONAL LAYER
# ============================================================

LAST_CONV_LAYER = "top_conv"

In [ ]:
# ============================================================
# 7. GRAD-CAM FUNCTION
# ============================================================

def make_gradcam_heatmap(
    img_array,
    model,
    last_conv_layer_name
):

    # --------------------------------------------------------
    # GET BACKBONE
    # --------------------------------------------------------

    backbone = model.layers[0]

    # --------------------------------------------------------
    # GET LAST CONVOLUTIONAL LAYER
    # --------------------------------------------------------

    last_conv_layer = backbone.get_layer(
        last_conv_layer_name
    )

    # --------------------------------------------------------
    # FEATURE EXTRACTOR
    # --------------------------------------------------------

    feature_extractor = tf.keras.Model(
        inputs=backbone.inputs,
        outputs=last_conv_layer.output
    )

    # --------------------------------------------------------
    # CLASSIFIER MODEL
    # --------------------------------------------------------

    classifier_input = tf.keras.Input(
        shape=last_conv_layer.output.shape[1:]
    )

    x = classifier_input

    x = tf.keras.layers.GlobalAveragePooling2D()(x)

    classifier_output = model.layers[-1](x)

    classifier_model = tf.keras.Model(
        classifier_input,
        classifier_output
    )

    # --------------------------------------------------------
    # COMPUTE GRADIENTS
    # --------------------------------------------------------

    with tf.GradientTape() as tape:

        conv_outputs = feature_extractor(
            img_array
        )

        tape.watch(conv_outputs)

        predictions = classifier_model(
            conv_outputs
        )

        pred_index = tf.argmax(
            predictions[0]
        )

        class_channel = predictions[
            :,
            pred_index
        ]

    # --------------------------------------------------------
    # COMPUTE GRADIENTS
    # --------------------------------------------------------

    grads = tape.gradient(
        class_channel,
        conv_outputs
    )

    # --------------------------------------------------------
    # CHANNEL IMPORTANCE
    # --------------------------------------------------------

    pooled_grads = tf.reduce_mean(
        grads,
        axis=(0,1,2)
    )

    # --------------------------------------------------------
    # REMOVE BATCH DIMENSION
    # --------------------------------------------------------

    conv_outputs = conv_outputs[0]

    # --------------------------------------------------------
    # STANDARD GRAD-CAM
    # --------------------------------------------------------

    heatmap = tf.reduce_sum(
        conv_outputs * pooled_grads,
        axis=-1
    )

    # --------------------------------------------------------
    # RELU
    # --------------------------------------------------------

    heatmap = tf.maximum(
        heatmap,
        0
    )

    # --------------------------------------------------------
    # NORMALIZE
    # --------------------------------------------------------

    max_val = tf.reduce_max(
        heatmap
    )

    if max_val > 0:

        heatmap /= max_val

    # --------------------------------------------------------
    # SHARPEN HEATMAP
    # --------------------------------------------------------

    heatmap = tf.where(
        heatmap < 0.6,
        0.0,
        heatmap
    )

    return heatmap.numpy()

In [ ]:
# ============================================================
# 8. OVERLAY FUNCTION
# ============================================================

def overlay_heatmap(
    heatmap,
    image,
    alpha=0.15
):

    # --------------------------------------------------------
    # RESIZE HEATMAP
    # --------------------------------------------------------

    heatmap = cv2.resize(
        heatmap,
        (image.shape[1], image.shape[0])
    )

    # --------------------------------------------------------
    # NORMALIZE
    # --------------------------------------------------------

    heatmap = np.uint8(
        255 * heatmap
    )

    # --------------------------------------------------------
    # BETTER COLORMAP
    # --------------------------------------------------------

    heatmap = cv2.applyColorMap(
        heatmap,
        cv2.COLORMAP_TURBO
    )

    # --------------------------------------------------------
    # OVERLAY
    # --------------------------------------------------------

    overlay = cv2.addWeighted(
        image,
        1-alpha,
        heatmap,
        alpha,
        0
    )

    overlay = cv2.cvtColor(
        overlay,
        cv2.COLOR_BGR2RGB
    )

    return overlay

In [ ]:
# ============================================================
# 9. GET IMAGE FILES
# ============================================================

image_files = [

    file for file in os.listdir(
        test_folder
    )

    if file.lower().endswith(
        ('.jpg','.jpeg','.png')
    )
]

print("\nImages Found:")
print(image_files)



Images Found:
['gradcam_Train_26.jpg', 'gradcam_Train_83.jpg', 'gradcam_Train_29.jpg']


In [ ]:
# ============================================================
# 10. PROCESS ALL IMAGES
# ============================================================

# ------------------------------------------------------------
# DELETE OLD RESULTS
# ------------------------------------------------------------

import shutil

if os.path.exists("/content/gradcam_results"):

    shutil.rmtree("/content/gradcam_results")

os.makedirs(
    "/content/gradcam_results",
    exist_ok=True
)

print("Old results deleted")

# ============================================================
# PROCESS IMAGES
# ============================================================

for file_name in image_files:

    print(f"\nProcessing: {file_name}")

    img_path = os.path.join(
        test_folder,
        file_name
    )

    # --------------------------------------------------------
    # LOAD IMAGE
    # --------------------------------------------------------

    img = image.load_img(
        img_path,
        target_size=(IMG_SIZE, IMG_SIZE)
    )

    img_array = image.img_to_array(img)

    input_image = np.expand_dims(
        img_array,
        axis=0
    )

    input_image = preprocess_input(
        input_image
    )

    # --------------------------------------------------------
    # ORIGINAL IMAGE
    # --------------------------------------------------------

    original_img = cv2.imread(
        img_path
    )

    original_img = cv2.resize(
        original_img,
        (IMG_SIZE, IMG_SIZE)
    )

    original_rgb = cv2.cvtColor(
        original_img,
        cv2.COLOR_BGR2RGB
    )

    # --------------------------------------------------------
    # GENERATE HEATMAPS
    # --------------------------------------------------------

    baseline_heatmap = make_gradcam_heatmap(
        input_image,
        baseline_model,
        LAST_CONV_LAYER
    )

    kd_heatmap = make_gradcam_heatmap(
        input_image,
        student_model_pp2020,
        LAST_CONV_LAYER
    )

    # --------------------------------------------------------
    # OVERLAY HEATMAPS
    # --------------------------------------------------------

    baseline_overlay = overlay_heatmap(
        baseline_heatmap,
        original_img
    )

    kd_overlay = overlay_heatmap(
        kd_heatmap,
        original_img
    )

    # --------------------------------------------------------
    # PREDICTIONS
    # --------------------------------------------------------

    baseline_pred = baseline_model.predict(
        input_image,
        verbose=0
    )

    kd_pred = student_model_pp2020.predict(
        input_image,
        verbose=0
    )

    baseline_class = class_names[
        np.argmax(baseline_pred)
    ]

    kd_class = class_names[
        np.argmax(kd_pred)
    ]

    # ========================================================
    # VISUALIZATION
    # ========================================================

    fig = plt.figure(
        figsize=(18,6)
    )

    # --------------------------------------------------------
    # ORIGINAL IMAGE
    # --------------------------------------------------------

    plt.subplot(1,3,1)

    plt.imshow(original_rgb)

    plt.title(
        "Original Image",
        fontsize=16
    )

    plt.axis("off")

    # --------------------------------------------------------
    # BASELINE STUDENT
    # --------------------------------------------------------

    plt.subplot(1,3,2)

    plt.imshow(baseline_overlay)

    plt.title(
        f"Baseline Student\nPrediction: {baseline_class}",
        fontsize=16
    )

    plt.axis("off")

    # --------------------------------------------------------
    # KD STUDENT
    # --------------------------------------------------------

    plt.subplot(1,3,3)

    plt.imshow(kd_overlay)

    plt.title(
        f"KD Student\nPrediction: {kd_class}",
        fontsize=16
    )

    plt.axis("off")

    # --------------------------------------------------------
    # BETTER SPACING
    # --------------------------------------------------------

    plt.subplots_adjust(
        wspace=0.05,
        hspace=0
    )

    # ========================================================
    # SAVE RESULT
    # ========================================================

    save_path = os.path.join(
        save_dir,
        f"gradcam_{file_name}"
    )

    fig.tight_layout()

    fig.savefig(
        save_path,
        dpi=300,
        facecolor='white'
    )

    plt.close(fig)
    print(f"Saved: {save_path}")

# ============================================================
# ZIP RESULTS
# ============================================================

zip_path = "/content/gradcam_results.zip"

shutil.make_archive(
    "/content/gradcam_results",
    'zip',
    save_dir
)

print("\nZIP CREATED:")
print(zip_path)

# ============================================================
# DOWNLOAD ZIP
# ============================================================

from google.colab import files

files.download(
    zip_path
)

print("\n================================================")
print("GRAD-CAM ANALYSIS COMPLETED")
print("================================================")

Old results deleted

Processing: gradcam_Train_26.jpg


/usr/local/lib/python3.12/dist-packages/keras/src/models/functional.py:241: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: ['keras_tensor']
Received: inputs=Tensor(shape=(1, 380, 380, 3))
  warnings.warn(msg)
/usr/local/lib/python3.12/dist-packages/keras/src/models/functional.py:241: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: ['keras_tensor_481']
Received: inputs=Tensor(shape=(1, 380, 380, 3))
  warnings.warn(msg)


Saved: /content/gradcam_images/gradcam_gradcam_Train_26.jpg

Processing: gradcam_Train_83.jpg


/usr/local/lib/python3.12/dist-packages/keras/src/models/functional.py:241: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: ['keras_tensor']
Received: inputs=Tensor(shape=(1, 380, 380, 3))
  warnings.warn(msg)
/usr/local/lib/python3.12/dist-packages/keras/src/models/functional.py:241: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: ['keras_tensor_481']
Received: inputs=Tensor(shape=(1, 380, 380, 3))
  warnings.warn(msg)


Saved: /content/gradcam_images/gradcam_gradcam_Train_83.jpg

Processing: gradcam_Train_29.jpg


/usr/local/lib/python3.12/dist-packages/keras/src/models/functional.py:241: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: ['keras_tensor']
Received: inputs=Tensor(shape=(1, 380, 380, 3))
  warnings.warn(msg)
/usr/local/lib/python3.12/dist-packages/keras/src/models/functional.py:241: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: ['keras_tensor_481']
Received: inputs=Tensor(shape=(1, 380, 380, 3))
  warnings.warn(msg)


Saved: /content/gradcam_images/gradcam_gradcam_Train_29.jpg

ZIP CREATED:
/content/gradcam_results.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


GRAD-CAM ANALYSIS COMPLETED
